# Notebook 3: RAG Pipeline — Zillow Data Retrieval
## Iowa City Housing LLM

**Prerequisites:** Complete `01_environment_setup.ipynb` and `02_finetune_llama3.ipynb`.

This notebook builds the **Retrieval-Augmented Generation (RAG)** layer that:
1. Embeds your Zillow JSONL documents into a FAISS vector index
2. At query time, retrieves the most relevant chunks from the index
3. Injects retrieved context into the prompt before generating a response
4. Saves the vector index for reuse

---
### Why RAG on top of fine-tuning?
- Fine-tuning bakes in Iowa City housing **knowledge and conversational style**
- RAG provides **current, updatable data** (today's Zillow prices, new listings, etc.)
- As new Zillow data arrives, you only re-index — no re-training required

## 1. Imports & Config

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [2]:
import os
import json
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

RAG_CONFIG = {
    "raw_rag_data": "../MarketMindAI/housing_rag_zillow.jsonl",
    "rag_index_path": "../models/faiss_rag_index",
    "finetuned_model_path": "SiwgiLi/llama3-housing-lora-merged",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "chunk_size": 512,
    "chunk_overlap": 64,
    "top_k_docs": 3,
    "similarity_threshold": 0.3,
    "max_new_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.9,
}

os.makedirs(RAG_CONFIG["rag_index_path"], exist_ok=True)
print("Config ready")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Config ready


## 2. Load & Chunk the Zillow JSONL Data

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

def load_rag_documents(jsonl_path):
    """Load Zillow JSONL and convert to LangChain Documents."""
    documents = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            metadata = {
                "id": record.get("id", ""),
                "source": record.get("source_name", ""),
                "url": record.get("source_url", ""),
                "date": record.get("publication_date", ""),
                "section": record.get("section", ""),
                "tags": ", ".join(record.get("topic_tags", [])),
            }
            doc = Document(page_content=record["text"], metadata=metadata)
            documents.append(doc)
    return documents

raw_docs = load_rag_documents(RAG_CONFIG["raw_rag_data"])
print(f"Loaded {len(raw_docs)} raw documents")

for doc in raw_docs:
    print(f"  [{doc.metadata['id']}] {doc.metadata['section']} ({len(doc.page_content)} chars)")

Loaded 3 raw documents
  [housing_zillow_001] Zillow Home Value Index (ZHVI) — Citywide Overview (993 chars)
  [housing_zillow_002] Zillow Neighborhood Breakdown — Iowa City (743 chars)
  [housing_zillow_003] Zillow Observed Rent Index (ZORI) — Iowa City (866 chars)


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=RAG_CONFIG["chunk_size"],
    chunk_overlap=RAG_CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", " ", ""],
)

split_docs = text_splitter.split_documents(raw_docs)

print(f"Split into {len(split_docs)} chunks from {len(raw_docs)} documents")
lengths = [len(doc.page_content) for doc in split_docs]
print(f"Chunk lengths — min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)//len(lengths)}")

print("\nSample chunk:")
print(split_docs[0].page_content[:300])

Split into 8 chunks from 3 documents
Chunk lengths — min: 178, max: 460, avg: 324

Sample chunk:
As of February 28, 2026, the Zillow Home Value Index (ZHVI) for Iowa City, IA is $292,103,
representing a 4.6 percent increase over the prior year. The ZHVI is built from monthly
changes in property-level Zestimates and is designed to capture both the level and
directional change of home values acro


## 3. Create Embeddings & Build FAISS Index

In [5]:
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading embedding model: {RAG_CONFIG['embedding_model']}")
embeddings = HuggingFaceEmbeddings(
    model_name=RAG_CONFIG["embedding_model"],
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

test_embedding = embeddings.embed_query("Iowa City home prices")
print(f"Embedding model loaded. Dimension: {len(test_embedding)}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


W0411 20:38:06.604000 37116 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Embedding model loaded. Dimension: 384


In [6]:
print(f"Building FAISS index from {len(split_docs)} chunks...")

vectorstore = FAISS.from_documents(documents=split_docs, embedding=embeddings)
vectorstore.save_local(RAG_CONFIG["rag_index_path"])

print(f"FAISS index saved to: {RAG_CONFIG['rag_index_path']}")
print(f"Total vectors: {vectorstore.index.ntotal}")

Building FAISS index from 8 chunks...
FAISS index saved to: ../models/faiss_rag_index
Total vectors: 8


## 4. Test the Retrieval System

In [7]:
def retrieve_context(query, vectorstore, top_k=3, threshold=0.3):
    """Retrieve the most relevant document chunks for a query."""
    results = vectorstore.similarity_search_with_relevance_scores(query=query, k=top_k)
    retrieved = []
    for doc, score in results:
        if score >= threshold:
            retrieved.append({
                "content": doc.page_content,
                "score": score,
                "source": doc.metadata.get("source", "Unknown"),
                "section": doc.metadata.get("section", ""),
                "date": doc.metadata.get("date", ""),
                "url": doc.metadata.get("url", ""),
            })
    return retrieved


test_queries = [
    "What is the average rent in Iowa City?",
    "Which Iowa City neighborhoods have the highest home values?",
    "How many homes are currently listed for sale in Iowa City?",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    results = retrieve_context(query, vectorstore, top_k=RAG_CONFIG["top_k_docs"])
    for i, r in enumerate(results):
        print(f"  [{i+1}] Score: {r['score']:.3f} | {r['section']} ({r['date']})")
        print(f"       {r['content'][:120]}...")


Query: What is the average rent in Iowa City?
  [1] Score: 0.685 | Zillow Observed Rent Index (ZORI) — Iowa City (2026-02-28)
       Iowa City Rental Market (Zillow Observed Rent Index / ZORI, as of February 28, 2026):

 Average Rent (Iowa City): $1,308...
  [2] Score: 0.613 | Zillow Observed Rent Index (ZORI) — Iowa City (2026-02-28)
       Iowa City average rents are approximately 31 percent below the national average, reflecting
both Iowa's generally lower ...
  [3] Score: 0.365 | Zillow Neighborhood Breakdown — Iowa City (2026-02-28)
       Zillow Home Value Index (ZHVI) by Iowa City neighborhood (data through February 2026):

 South Pointe: $314,570 (highest...

Query: Which Iowa City neighborhoods have the highest home values?
  [1] Score: 0.512 | Zillow Neighborhood Breakdown — Iowa City (2026-02-28)
       Zillow Home Value Index (ZHVI) by Iowa City neighborhood (data through February 2026):

 South Pointe: $314,570 (highest...
  [2] Score: 0.457 | Zillow Home Value Index (ZHV

## 5. Build the Full RAG + LLM Pipeline

In [8]:
from transformers import pipeline as hf_pipeline

SYSTEM_PROMPT_RAG = """You are an Iowa City Housing Assistant specializing in ZIP codes 
52245 and 52242. You help recent college graduates and young families make informed 
housing decisions using current local market data.

You will be given CONTEXT from the latest Zillow data for Iowa City. Use this context 
to provide specific, data-driven answers. Always cite the data source and date when 
referencing specific numbers. Always clarify that your guidance is educational and 
not professional financial advice."""


def format_rag_prompt(question, retrieved_docs):
    if not retrieved_docs:
        context_text = "No specific current data found. Providing general guidance."
    else:
        context_parts = []
        for i, doc in enumerate(retrieved_docs):
            context_parts.append(
                f"[Source {i+1}: {doc['source']}, {doc['date']}, {doc['section']}]\n{doc['content']}"
            )
        context_text = "\n\n".join(context_parts)
    return f"CONTEXT (from current Zillow data):\n{context_text}\n\nUSER QUESTION: {question}\n\nPlease answer using the context above where relevant."


class IowaCityHousingRAG:
    """Full RAG pipeline: FAISS retrieval + fine-tuned LLaMA 3 generation."""

    def __init__(self, model_path, rag_index_path, embedding_model):
        print("Loading RAG components...")
        emb = HuggingFaceEmbeddings(
            model_name=embedding_model,
            model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )
        self.vectorstore = FAISS.load_local(rag_index_path, emb, allow_dangerous_deserialization=True)
        print(f"Loading model from: {model_path}")
        self.pipe = hf_pipeline(
            "text-generation",
            model=model_path,
            tokenizer=model_path,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        self.pipe.tokenizer.pad_token_id = self.pipe.tokenizer.eos_token_id
        print("RAG pipeline ready!")

    def answer(self, question, top_k=3, max_new_tokens=512, verbose=False):
        retrieved_docs = retrieve_context(question, self.vectorstore, top_k=top_k)
        if verbose:
            print(f"Retrieved {len(retrieved_docs)} chunks")
            for doc in retrieved_docs:
                print(f"  [{doc['score']:.3f}] {doc['section']}")
        augmented_question = format_rag_prompt(question, retrieved_docs)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_RAG},
            {"role": "user", "content": augmented_question},
        ]
        output = self.pipe(
            messages,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=self.pipe.tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
        response = output[0]["generated_text"][-1]["content"]
        return {
            "question": question,
            "answer": response,
            "sources": [{"section": d["section"], "date": d["date"], "score": d["score"]} for d in retrieved_docs]
        }


rag_pipeline = IowaCityHousingRAG(
    model_path=RAG_CONFIG["finetuned_model_path"],
    rag_index_path=RAG_CONFIG["rag_index_path"],
    embedding_model=RAG_CONFIG["embedding_model"],
)

Loading RAG components...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model from: SiwgiLi/llama3-housing-lora-merged


Device set to use cuda:0


RAG pipeline ready!


## 6. Test the Full RAG Pipeline

In [9]:
rag_test_questions = [
    "What is the current average rent in Iowa City?",
    "How much have home values increased in Iowa City this year?",
    "Which Iowa City neighborhood should I look at as a first-time buyer under $250,000?",
    "Is it currently a buyer's or seller's market in Iowa City?",
]

for question in rag_test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print('='*60)
    result = rag_pipeline.answer(question, verbose=True)
    print(f"\nA: {result['answer']}")
    print(f"\nSources: {result['sources']}")


Q: What is the current average rent in Iowa City?
Retrieved 3 chunks
  [0.724] Zillow Observed Rent Index (ZORI) — Iowa City
  [0.636] Zillow Observed Rent Index (ZORI) — Iowa City
  [0.375] Zillow Neighborhood Breakdown — Iowa City

A: Based on the provided Zillow data, the current average rent in Iowa City, as of February 28, 2026, is $1,308 per month.

This information comes directly from Source 1: Zillow: Iowa City, IA Housing Market — ZHVI & ZORI, 2026-02-28, Zillow Observed Rent Index (ZORI) — Iowa City, which reports:

"Average Rent (Iowa City): $1,308/month"

Additionally, Source 2: Zillow: Iowa City, IA Housing Market — ZHVI & ZORI, 2026-02-28, Zillow Observed Rent Index (ZORI) — Iowa City, provides a monthly rent figure of $1,308 for Iowa City, indicating it remains one of the lowest prices in the city.

These data points confirm that the current average rent in Iowa City is indeed $1,308 per month.

Sources: [{'section': 'Zillow Observed Rent Index (ZORI) — Iowa City', 'dat

## 7. Updating the Index with New Data

In [10]:
def add_new_data_to_index(new_jsonl_path, existing_vectorstore, save_path):
    """
    Add new Zillow JSONL data to an existing FAISS index.
    Call this monthly when you receive updated Zillow data.
    No model re-training needed — just re-index!
    """
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    new_docs = load_rag_documents(new_jsonl_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
    new_chunks = splitter.split_documents(new_docs)
    existing_vectorstore.add_documents(new_chunks)
    existing_vectorstore.save_local(save_path)
    print(f"Added {len(new_chunks)} chunks. Total vectors: {existing_vectorstore.index.ntotal}")
    return existing_vectorstore


# USAGE — run monthly with new data:
# updated_vs = add_new_data_to_index(
#     "../data/housing_rag_zillow_march2026.jsonl",
#     rag_pipeline.vectorstore,
#     RAG_CONFIG["rag_index_path"]
# )
print("add_new_data_to_index() defined. Call with new monthly Zillow data files.")

add_new_data_to_index() defined. Call with new monthly Zillow data files.


## 8. ZIP Code Expansion Roadmap

In [11]:
EXPANSION_PLAN = {
    "Phase 1 (Active)": {"zip_codes": ["52245", "52242"], "note": "Iowa City core — current scope"},
    "Phase 2 (Planned)": {"zip_codes": ["52240", "52246"], "note": "Full Iowa City — add data + re-index only"},
    "Phase 3 (Future)": {"zip_codes": ["52241", "52302"], "note": "Coralville + Marion — may require new fine-tuning"},
    "Phase 4 (Long-term)": {"zip_codes": ["Iowa statewide"], "note": "All major Iowa metros — city-specific LoRA adapters"},
}

for phase, info in EXPANSION_PLAN.items():
    print(f"{phase}: {info['zip_codes']}")
    print(f"  -> {info['note']}")

Phase 1 (Active): ['52245', '52242']
  -> Iowa City core — current scope
Phase 2 (Planned): ['52240', '52246']
  -> Full Iowa City — add data + re-index only
Phase 3 (Future): ['52241', '52302']
  -> Coralville + Marion — may require new fine-tuning
Phase 4 (Long-term): ['Iowa statewide']
  -> All major Iowa metros — city-specific LoRA adapters


## Summary

The RAG pipeline is complete:
- Zillow JSONL documents embedded and indexed in FAISS
- Retrieval tested and working with relevance scoring
- Full `IowaCityHousingRAG` pipeline assembled (retrieve + generate)
- `add_new_data_to_index()` ready for monthly Zillow data updates
- Expansion roadmap documented

**Next:** Run `04_local_testing.ipynb` for the interactive chat UI and full evaluation.